In [1]:
import os
import random
import numpy as np
import sys
sys.path.append( '/home/wbguo/iproject/BSReadSim')
import subprocess

from Bio import SeqIO
from tqdm import tqdm
from scipy.stats import bernoulli
from typing import Dict, Union, Tuple

from SetCyotsineMethylation import SetCytosineMethylation
from StreamSim import StreamWGSIM
from Utils.UtilityFunctions import get_external_paths, reverse_complement
from SimulateMethylatedReads import SimulateMethylatedReads

In [2]:
reference_file = "/home/wbguo/iproject/BSReadSim/TestData/test2/BSB_test.fa"
sim_output = "/home/wbguo/iproject/BSReadSim/TestData/test2/sim"
vcf_file = "/home/wbguo/iproject/BSReadSim/TestData/test2/test.vcf"
cgmap_file = "/home/wbguo/iproject/BSReadSim/TestData/test2/test.CGmap.gz"
simulation = SimulateMethylatedReads(reference_file=reference_file, sim_output=sim_output, vcf_file=vcf_file, cgmap_file = cgmap_file)

Must compile external dependencies
 python3 setup.py build


In [3]:
simulation.run()

Generating methylation profile:



93087it [00:00, 138776.59it/s]
94321it [00:00, 148133.11it/s]
94030it [00:00, 147762.95it/s]
93814it [00:00, 154666.69it/s]
91435it [00:00, 149626.21it/s]
92814it [00:00, 149705.04it/s]
70773it [00:00, 146819.79it/s]
71302it [00:00, 155017.03it/s]
76646it [00:00, 150312.14it/s]
77512it [00:00, 155546.83it/s]
1132it [00:00, 139477.46it/s]
1120it [00:00, 131877.84it/s]


Simulating methylated Reads:

[CMD]: /home/wbguo/iproject/BSReadSim/External/WGSIM/wgsim /home/wbguo/iproject/BSReadSim/TestData/test2/BSB_test.fa -1 100 -2 100 -N 196160 -r 0.001 -h 0 -S -1 -R 0.15 -X 0.15 -d 400 -s 25 -I 100 -e 0 -A 0.05 -g /home/wbguo/iproject/BSReadSim/TestData/test2/test.vcf



Simulating Bisulfite Converted Read Pairs: 0it [00:00, ?it/s][wgsim] seed = 1660272008
[wgsim_core] calculating the total length of the reference sequence...
[wgsim_core] 6 sequences, total length: 1961600
VCF file exists, use it to simulate reads
[parse_vcf_chr] Finish collecting 360 SNP from chr10
Simulating Bisulfite Converted Read Pairs: 42269it [04:02, 174.14it/s][parse_vcf_chr] Finish collecting 309 SNP from chr11
Simulating Bisulfite Converted Read Pairs: 84677it [08:04, 176.07it/s][parse_vcf_chr] Finish collecting 315 SNP from chr12
Simulating Bisulfite Converted Read Pairs: 126853it [12:02, 183.03it/s][parse_vcf_chr] Finish collecting 264 SNP from chr13
Simulating Bisulfite Converted Read Pairs: 160467it [15:08, 180.77it/s][parse_vcf_chr] Finish collecting 300 SNP from chr14
Simulating Bisulfite Converted Read Pairs: 195591it [18:29, 178.12it/s][parse_vcf_chr] Finish collecting 0 SNP from chr15
Simulating Bisulfite Converted Read Pairs: 196166it [18:32, 176.30it/s]

Simulation Finished!


In [3]:
reference_dict = SeqIO.to_dict(SeqIO.parse(reference_file, "fasta"))

In [4]:
meth_beta_param: Dict = {"CG": (0.5, 0.5), "CHG": (0.01, 0.05), "CHH":(0.01, 0.05)}
mutation_rate: float = 0.0010
haplotype_mode: bool = False
random_seed: int = -1
mutation_indel_fraction: float = 0.15
indel_extension_probability: float = 0.15
pe_fragment_size: int = 400
insert_deviation: int = 25
mean_inner_dist: int = 100
read_length: int = 100
read_depth: int = 20
sequencing_error: float = 0.005
undirectional: bool = False
paired_end: bool = True
conversion_rate: float = 0.998
ambiguous_base_cutoff: float = 0.05
verbose: bool = True
collect_ch_sites: bool = True
collect_sim_stats: bool = False
overwrite_db: bool = False

In [5]:
if not os.path.exists(reference_file):
    raise ValueError('Cannot find the reference file, please check!')
if not sim_output:
    raise ValueError('Please specify the output directory!')

In [6]:
wgsim_args     = [get_external_paths()[1], reference_file] # second element is wgsim path
genome_length  = sum([len(seq) for key, seq in reference_dict.items()])
number_reads   = int(genome_length * read_depth / read_length / (1 + int(paired_end)))
wgsim_options  = {'-1': read_length, '-2': read_length, '-N': number_reads,
                  '-r': mutation_rate, '-h': int(haplotype_mode), '-S': random_seed, 
                  '-R': mutation_indel_fraction,'-X': indel_extension_probability,
                  '-d': pe_fragment_size, '-s': insert_deviation, '-I': mean_inner_dist,
                  '-e': 0, '-A': ambiguous_base_cutoff, '-g': vcf_file} #set -e to be 0
wgsim_args     = wgsim_args
wgsim_options  = wgsim_options

Must compile external dependencies
 python3 setup.py build


In [7]:
wgsim_options

{'-1': 100,
 '-2': 100,
 '-N': 196160,
 '-r': 0.001,
 '-h': 0,
 '-S': -1,
 '-R': 0.15,
 '-X': 0.15,
 '-d': 400,
 '-s': 25,
 '-I': 100,
 '-e': 0,
 '-A': 0.05,
 '-g': '/home/wbguo/iproject/BSReadSim/TestData/test2/test.vcf'}

In [8]:
meth_options   = {'cgmap': cgmap_file, 'meth_ref': None, 'beta_param': meth_beta_param}

In [9]:
sim_meth_db = SetCytosineMethylation(reference_file=reference_file,
                                     sim_output=sim_output,
                                     meth_ref=meth_options['meth_ref'],
                                     cgmap=meth_options['cgmap'],
                                     beta_param = meth_options['beta_param'],
                                     collect_ch_sites=collect_ch_sites,
                                     overwrite_db=overwrite_db)

93087it [00:00, 142802.55it/s]
94321it [00:00, 149406.00it/s]
94030it [00:00, 149140.78it/s]
93814it [00:00, 155569.07it/s]
91435it [00:00, 151134.97it/s]
92814it [00:00, 148839.66it/s]
70773it [00:00, 151326.84it/s]
71302it [00:00, 156774.92it/s]
76646it [00:00, 152346.10it/s]
77512it [00:00, 155925.96it/s]
1132it [00:00, 140596.75it/s]
1120it [00:00, 139391.13it/s]


In [10]:
sub_pattern = ('C', 'T')

In [11]:
sim_meth_db.profile_dict['chr10'][5015]

array([1., 0., 0.])

In [12]:
sim_meth_db

In [13]:
sim_command = wgsim_args + [str(item) for key_val in wgsim_options.items() for item in key_val]

In [16]:
sim_command

['/home/wbguo/iproject/BSReadSim/External/WGSIM/wgsim',
 '/home/wbguo/iproject/BSReadSim/TestData/test2/BSB_test.fa',
 '-1',
 '100',
 '-2',
 '100',
 '-N',
 '196160',
 '-r',
 '0.001',
 '-h',
 '0',
 '-S',
 '-1',
 '-R',
 '0.15',
 '-X',
 '0.15',
 '-d',
 '400',
 '-s',
 '25',
 '-I',
 '100',
 '-e',
 '0',
 '-A',
 '0.05',
 '-g',
 '/home/wbguo/iproject/BSReadSim/TestData/test2/test.vcf']

In [28]:
wgsim = subprocess.Popen(sim_command, stdout=subprocess.PIPE, universal_newlines=True)
sim_iter = iter(wgsim.stdout.readline, b'')

[wgsim] seed = 1660429769
[wgsim_core] calculating the total length of the reference sequence...
[wgsim_core] 6 sequences, total length: 1961600
VCF file exists, use it to simulate reads
[parse_vcf_chr] Finish collecting 360 SNP from chr10


In [29]:
sim_iter

In [30]:
def get_line(sim_iter):
    try:
        line = next(sim_iter).strip()
    except StopIteration:
        print("End of output\n")
        return None
    else:
        return line

In [31]:
line = get_line(sim_iter)

In [32]:
line

'Contig Variant Start'

In [33]:
line = get_line(sim_iter)

In [34]:
line

'chr10\t4943\tT\tA\t-'

In [35]:
line = get_line(sim_iter)


In [36]:
line

'chr10\t5160\tC\tY\t+'

In [15]:
def get_methylation_reference(contig: str, variant_data:  Dict= False) -> Dict:
        """ Set variant methylation if variant data provided and return methylation profile else
        return methylation profile

        Params:

        * *contig (str)*: contig id
        * *variant_data (dict)*: simulated variant information

        Returns:

        * *contig_profile (Dict[str, float])*: methylation reference values"""
        #if variant_data:
        contig_profile = sim_meth_db.get_contig_methylation(contig)
            # self.sim_meth_db.set_variant_methylation(variant_data, self.contig_profile, self.current_contig)
        return contig_profile
        #return self.sim_meth_db.get_contig_methylation(contig)

def process_read_group(sim_data, stranded_capture=False):
        """Set read methylation values, randomly assign reads to Watson or Crick strand, bisulfite conversion"""
        ref_strand  = 'W' # watson
        sub_pattern = ('C', 'T')
        strand_swtich = (not stranded_capture) and bernoulli.rvs(0.5) # randomly select reference strand
        
        if strand_swtich:
            ref_strand  = 'C' # crick
            sub_pattern = ('G', 'A')
            sim_data[0], sim_data[1] = sim_data[1], sim_data[0]                         #??? need to check
        
        # set read methylation, bisulfite conversion, add sequencing error
        set_read_methylation_bisulfite(sim_data[0], sub_pattern)
        


        
        
for variant_contig, sim_data in tqdm(StreamWGSIM(sim_command=sim_command),
                                             desc='Simulating Bisulfite Converted Read Pairs',
                                             position=0, leave=True):
            if variant_contig:
                contig_profile = get_methylation_reference(variant_contig, sim_data) #sim_data can be none
            else:
                #process_read_group(sim_data)
                print(sim_data)

Simulating Bisulfite Converted Read Pairs: 0it [00:00, ?it/s][wgsim] seed = 1660415275
[wgsim_core] calculating the total length of the reference sequence...
[wgsim_core] 6 sequences, total length: 1961600
VCF file exists, use it to simulate reads
[parse_vcf_chr] Finish collecting 360 SNP from chr10
Simulating Bisulfite Converted Read Pairs: 1it [00:00,  1.93it/s]


TypeError: list indices must be integers or slices, not str

In [4]:
sim_data

NameError: name 'sim_data' is not defined

In [18]:
read = sim_data[0]

In [19]:
read_seq = read[1]
read_info= read[0]

In [20]:
seq_array = np.array(list(read_seq))

In [21]:
cigar_array = np.array(list(read_info['cigar']))

In [22]:
read_info['start']

226160

In [23]:
sub_pattern = ('C', 'T')

In [24]:
sub_base = sub_pattern[0]

In [25]:
meth_base_info = read_info['c_base_info']

In [26]:
ref_pos_dict= dict()
# if there is methylable base
if len(meth_base_info):
    seq_pos_offset_array = np.array([site.split("_") for site in meth_base_info.split(",")[:-1]]).transpose().astype(int)
    seq_pos_array = seq_pos_offset_array[0]                    # methylable base position on the read
    offset_array  = seq_pos_offset_array[1]
    ref_pos_array = read_info['start'] + seq_pos_array + offset_array  # methylable base position w.r.t the chromosome
    ref_pos_dict  = dict(zip(seq_pos_array, ref_pos_array))

In [27]:
cigar_M_idx = np.where(cigar_array == "M")[0] # match
cigar_X_idx = np.where(cigar_array == "X")[0] # substitution
cigar_I_idx = np.where(cigar_array == "I")[0] # insertion
seq_pos_array  = np.array(list(ref_pos_dict.keys()))

In [28]:
cigar_M_cg_idx = np.intersect1d(cigar_M_idx, seq_pos_array)

In [29]:
cigar_M_cg_idx

array([ 3, 12, 14, 20, 24, 27, 35, 38, 39, 43, 45, 47, 54, 59, 61, 63, 65,
       73, 74, 77, 78, 98])

In [30]:
idx = cigar_M_cg_idx[1]
meth_pos = ref_pos_dict[idx]

In [31]:
meth_pos

226172

In [36]:
def set_base_methylation(meth_pos) -> Tuple[str, str]:
    """set methylation satus for the matched methyable base"""
    # nucleotide, methylation_level, context, 0, 0, 0
    try:
        meth_info = contig_profile[meth_pos]
    except KeyError:
        return False, 0
    meth_status = bernoulli.rvs(meth_info[2])
    meth_context= meth_info[1]
    return meth_status, meth_context

In [39]:
for idx in cigar_M_cg_idx:
    meth_pos = ref_pos_dict[idx]
    meth_status, meth_context = set_base_methylation(meth_pos)
    if meth_status:
        cigar_status = 'C' if meth_context == "CG" else "Y"
    else:
        cigar_status = 'c' if meth_context == "CG" else "y"
    cigar_array[idx] = cigar_status

In [40]:
cigar_array

array(['M', 'M', 'M', 'Y', 'M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', 'y',
       'M', 'y', 'M', 'M', 'M', 'M', 'M', 'Y', 'M', 'M', 'M', 'y', 'M',
       'M', 'y', 'M', 'M', 'M', 'M', 'M', 'M', 'M', 'y', 'M', 'M', 'y',
       'y', 'M', 'M', 'M', 'y', 'M', 'y', 'M', 'y', 'M', 'M', 'M', 'M',
       'M', 'M', 'Y', 'M', 'M', 'M', 'M', 'y', 'M', 'Y', 'M', 'y', 'M',
       'y', 'M', 'M', 'M', 'M', 'M', 'M', 'M', 'y', 'y', 'M', 'M', 'y',
       'Y', 'M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', 'M',
       'M', 'M', 'M', 'M', 'M', 'M', 'M', 'Y', 'M'], dtype='<U1')

In [41]:
candidate_pos_cg = np.where(cigar_array == "c")[0]
candidate_pos_ch = np.where(cigar_array == "y")[0]
candidate_pos = np.append(candidate_pos_cg, candidate_pos_ch)

cigar_array[candidate_pos_cg] = "b"
cigar_array[candidate_pos_ch] = "d"

conversion_rate = 0.998
converted_res = [sub_pattern[i] for i in bernoulli.rvs(conversion_rate, size=len(candidate_pos))]
seq_array[candidate_pos] = converted_res

In [44]:
np.array(list(read_seq))

array(['T', 'G', 'T', 'C', 'T', 'T', 'T', 'A', 'T', 'G', 'T', 'G', 'C',
       'T', 'C', 'T', 'T', 'T', 'T', 'T', 'C', 'T', 'T', 'T', 'C', 'T',
       'T', 'C', 'T', 'T', 'T', 'G', 'G', 'T', 'T', 'C', 'T', 'T', 'C',
       'C', 'G', 'A', 'T', 'C', 'A', 'C', 'T', 'C', 'T', 'G', 'A', 'A',
       'G', 'T', 'C', 'T', 'G', 'T', 'T', 'C', 'T', 'C', 'T', 'C', 'T',
       'C', 'T', 'G', 'A', 'T', 'T', 'T', 'A', 'C', 'C', 'T', 'G', 'C',
       'C', 'T', 'G', 'T', 'G', 'G', 'A', 'G', 'T', 'T', 'T', 'G', 'G',
       'G', 'A', 'A', 'G', 'G', 'A', 'A', 'C', 'A'], dtype='<U1')

In [42]:
seq_array

array(['T', 'G', 'T', 'C', 'T', 'T', 'T', 'A', 'T', 'G', 'T', 'G', 'T',
       'T', 'T', 'T', 'T', 'T', 'T', 'T', 'C', 'T', 'T', 'T', 'T', 'T',
       'T', 'T', 'T', 'T', 'T', 'G', 'G', 'T', 'T', 'T', 'T', 'T', 'T',
       'T', 'G', 'A', 'T', 'T', 'A', 'T', 'T', 'T', 'T', 'G', 'A', 'A',
       'G', 'T', 'C', 'T', 'G', 'T', 'T', 'T', 'T', 'C', 'T', 'C', 'T',
       'T', 'T', 'G', 'A', 'T', 'T', 'T', 'A', 'T', 'T', 'T', 'G', 'T',
       'C', 'T', 'G', 'T', 'G', 'G', 'A', 'G', 'T', 'T', 'T', 'G', 'G',
       'G', 'A', 'A', 'G', 'G', 'A', 'A', 'C', 'A'], dtype='<U1')